In [1]:
# Cell 1: imports and paths
# Cell 2: channel mapping confirmation (keep from before)
# Cell 3: load_subject_preprocessed function from processed_16ch
# Cell 4: compute PSD on preprocessed epochs
# Cell 5: compare 9 subjects across groups

In [31]:
from pathlib import Path
import mne
import numpy as np
import matplotlib.pyplot as plt
import json

from scipy import stats
import sys
import time

### Functions import

In [3]:
sys.path.insert(0, '../functions')
from models import DANN
from data_loading import prepare_dataloaders, load_subject_data_flexible
from metrics import evaluate

from explainability import (
    DANNClassifierWrapper, load_model_for_xai,
    fold_batchnorm, verify_folding,
    lrp_epsilon_conv2d, lrp_zplus_conv2d,
    lrp_linear, lrp_avgpool,
    compute_lrp, run_lrp_full,
    load_lrp_attributions,
    compute_per_class_accuracy_106,
    compute_deeplift, run_deeplift_full,
    load_deeplift_attributions,
    compute_shap, run_shap_full,
    load_shap_attributions,
    compute_ig, run_ig_full,
    load_ig_attributions,
    run_misclassified_single,
    run_and_save_misclassified
)

from xai_utils import (
    assign_score_groups, get_grouped_order,
    get_score_grouped_order, classify_failure_groups,
    compute_lrp_deeplift_agreement,
    compute_method_agreement,
    compute_within_group_consistency,
    print_consistency_results,
    ch16_list,
    build_binary_relevance_table,
    build_multiclass_relevance_table,
    get_top_bottom_channels
)

from xai_plots import (
    plot_group_mean_attribution,
    plot_stopping_criterion_consistency
)

### Config

In [30]:
# Paths
raw_path = Path("../Data/raw/physionet_mi_data/eeg-motor-movementimagery-dataset")
base_path = Path("..")
processed_path = base_path / "Data/processed"
results_path = base_path / "results"
ch16_path = base_path / "Data/processed_16ch"

processed_16ch_path = base_path / "Data/processed_16ch"
raw_path = base_path / "Data/raw/physionet_mi_data/eeg-motor-movementimagery-dataset"
processed_16ch_path = Path("../Data/processed_16ch")

In [5]:
# 106 subjects after removing S088, S092, S100 (incompatible sampling rate)
excluded = ['S088', 'S092', 'S100']

valid_subjects = []
for i in range(1, 110):
    subject_id = f'S{str(i).zfill(3)}'
    subject_file = processed_path / f'{subject_id}_processed.npz'
    if subject_file.exists() and subject_id not in excluded:
        valid_subjects.append(subject_id)

print(f"Valid subjects: {len(valid_subjects)}")
print(f"First five: {valid_subjects[:5]}")
print(f"Last five: {valid_subjects[-5:]}")

Valid subjects: 106
First five: ['S001', 'S002', 'S003', 'S004', 'S005']
Last five: ['S105', 'S106', 'S107', 'S108', 'S109']


In [6]:
ch16_list = ['C3', 'Cz', 'C4', 'FC3', 'FCz', 'CP3', 'CPz', 'CP4',
             'CP5', 'CP6', 'P3', 'P4', 'C5', 'C6', 'P5', 'P6']

In [7]:
# Subjects to compare
subjects_to_check = {
    'top': ['S007', 'S085'],
    'negative_transfer': ['S003', 'S005', 'S018'],
    'group_a': ['S006', 'S037'],
    'group_b': ['S020', 'S066'],
    'neither': ['S030', 'S053', 'S065']
}

# Build flat list and group lookup in defined order
group_order = ['top', 'negative_transfer', 'group_a', 'group_b', 'neither']
all_subjects_flat = []
subject_groups = {}
for group in group_order:
    for subj in subjects_to_check[group]:
        all_subjects_flat.append(subj)
        subject_groups[subj] = group

print("Subjects defined:")
for group in group_order:
    print(f"  {group}: {subjects_to_check[group]}")
print(f"Total: {len(all_subjects_flat)} subjects")

# 16 channel names in same order
ch16_names = ['C3', 'Cz', 'C4', 'FC3', 'FCz', 'CP3', 'CPz', 'CP4',
              'CP5', 'CP6', 'P3', 'P4', 'C5', 'C6', 'P5', 'P6']

channels_to_compare = ['C3', 'C4', 'CP3', 'CP4']
channel_indices = {ch: ch16_names.index(ch) for ch in channels_to_compare}

sfreq = 160  # sampling frequency

Subjects defined:
  top: ['S007', 'S085']
  negative_transfer: ['S003', 'S005', 'S018']
  group_a: ['S006', 'S037']
  group_b: ['S020', 'S066']
  neither: ['S030', 'S053', 'S065']
Total: 12 subjects


## Pre processed signal analysis

### Functions

In [8]:
def compute_psd_preprocessed(subject_id, tmin_idx=48, tmax_idx=160):
    """
    Load preprocessed 16ch npz, extract time window,
    compute PSD in mu and beta bands.
    tmin_idx=48: 0.3s * 160Hz
    tmax_idx=160: 1.0s * 160Hz
    """
    data = np.load(processed_16ch_path / f"{subject_id}_16ch.npz")
    X = data['data']  # (n_trials, 16, 480)

    # Extract time window 0.3 to 1.0s
    X_window = X[:, :, tmin_idx:tmax_idx]  # (n_trials, 16, 112)

    # Compute PSD using numpy FFT
    n_times = X_window.shape[2]
    freqs = np.fft.rfftfreq(n_times, d=1./sfreq)

    # FFT across time axis
    fft_vals = np.abs(np.fft.rfft(X_window, axis=2)) ** 2
    # Average across trials
    psd_mean = np.mean(fft_vals, axis=0)  # (16, n_freqs)

    # Band indices
    mu_idx = (freqs >= 8) & (freqs <= 13)
    beta_idx = (freqs >= 13) & (freqs <= 30)

    results = {}
    for ch_name, ch_idx in channel_indices.items():
        results[ch_name] = {
            'mu': float(np.mean(psd_mean[ch_idx, mu_idx])),
            'beta': float(np.mean(psd_mean[ch_idx, beta_idx]))
        }
    return results


In [9]:
def compute_psd_from_array(X, sfreq=160, tmin_idx=48, tmax_idx=160):
    """
    Compute PSD from preprocessed epoch array.
    X: (n_trials, n_channels, n_timepoints)
    Returns: psd_mean (n_channels, n_freqs), freqs
    """
    X_window = X[:, :, tmin_idx:tmax_idx]
    n_times = X_window.shape[2]
    freqs = np.fft.rfftfreq(n_times, d=1./sfreq)
    fft_vals = np.abs(np.fft.rfft(X_window, axis=2)) ** 2
    psd_mean = np.mean(fft_vals, axis=0)
    psd_std = np.std(fft_vals, axis=0)
    return psd_mean, psd_std, freqs


def get_band_power(psd, freqs, fmin, fmax):
    """Extract mean power in frequency band."""
    idx = (freqs >= fmin) & (freqs <= fmax)
    return float(np.mean(psd[idx]))


def get_band_power_std(psd_std, freqs, fmin, fmax):
    """Extract mean std in frequency band."""
    idx = (freqs >= fmin) & (freqs <= fmax)
    return float(np.mean(psd_std[idx]))



In [10]:
def compute_subject_signal_profile(subject_id):
    """
    Signal profile for one subject from preprocessed 16ch data.
    Direction 2: per class power
    Direction 3: trial variability (std)
    Direction 1: lateralisation ratio
    """
    data = np.load(processed_16ch_path / f"{subject_id}_16ch.npz")
    X = data['data']
    y = data['labels']

    X_left = X[y == 0]
    X_right = X[y == 1]

    results = {}

    for cls_name, X_cls in [('left', X_left), ('right', X_right)]:
        psd_mean, psd_std, freqs = compute_psd_from_array(X_cls)
        results[cls_name] = {}
        for ch_name, ch_idx in channel_indices.items():
            mu_power = get_band_power(psd_mean[ch_idx], freqs, 8, 13)
            beta_power = get_band_power(psd_mean[ch_idx], freqs, 13, 30)
            mu_std = get_band_power_std(psd_std[ch_idx], freqs, 8, 13)
            beta_std = get_band_power_std(psd_std[ch_idx], freqs, 13, 30)
            # Coefficient of variation: std/mean, measures relative variability
            mu_cv = mu_std / mu_power if mu_power > 0 else 0
            results[cls_name][ch_name] = {
                'mu_power': mu_power,
                'beta_power': beta_power,
                'mu_std': mu_std,
                'beta_std': beta_std,
                'mu_cv': mu_cv  # direction 3: relative variability
            }

        # Direction 1: lateralisation ratio
        c3_mu = results[cls_name]['C3']['mu_power']
        c4_mu = results[cls_name]['C4']['mu_power']
        results[cls_name]['lateralisation_ratio'] = (
            (c3_mu - c4_mu) / (c3_mu + c4_mu)
        )

    return results


#### compute_raw_psd

In [11]:
def compute_raw_psd(subject_id, sfreq=160):
    """
    Compute mu and beta band power from raw unfiltered signal.
    All 16 channels, both classes.
    Returns dict with power and lateralisation ratio per class.
    """
    # Load raw and pick all 16 channels
    raw = load_raw_no_filter(subject_id)

    # Build channel map for all 16
    import re
    ch_map = {}
    for ch_clean in ch16_list:
        pattern = re.compile(f'^{ch_clean}\\.+$', re.IGNORECASE)
        matches = [ch for ch in raw.ch_names if pattern.match(ch)]
        if matches:
            ch_map[ch_clean] = matches[0]

    raw.pick(list(ch_map.values()))

    # Get events
    events, _ = mne.events_from_annotations(raw, verbose=False)

    # Extract epochs 0.3 to 1.0s
    epochs = mne.Epochs(raw, events,
                        event_id=dict(T1=2, T2=3),
                        tmin=0.3, tmax=1.0,
                        baseline=None, preload=True,
                        verbose=False)

    X = epochs.get_data()  # (n_trials, 16, n_times)
    y = epochs.events[:, 2] - 2  # 0=left, 1=right

    n_times = X.shape[2]
    freqs = np.fft.rfftfreq(n_times, d=1./sfreq)
    mu_idx = (freqs >= 8) & (freqs <= 13)
    beta_idx = (freqs >= 13) & (freqs <= 30)

    results = {}
    for cls, cls_name in [(0, 'left'), (1, 'right')]:
        X_cls = X[y == cls]
        fft_vals = np.abs(np.fft.rfft(X_cls, axis=2)) ** 2
        psd_mean = np.mean(fft_vals, axis=0)  # (16, n_freqs)
        psd_std = np.std(fft_vals, axis=0)

        results[cls_name] = {}
        for ch_idx, ch_name in enumerate(ch16_list):
            mu = float(np.mean(psd_mean[ch_idx, mu_idx]))
            beta = float(np.mean(psd_mean[ch_idx, beta_idx]))
            mu_std = float(np.mean(psd_std[ch_idx, mu_idx]))
            mu_cv = mu_std / mu if mu > 0 else 0
            results[cls_name][ch_name] = {
                'mu': mu,
                'beta': beta,
                'mu_cv': mu_cv
            }

        # Lateralisation ratio C3 vs C4
        c3_mu = results[cls_name]['C3']['mu']
        c4_mu = results[cls_name]['C4']['mu']
        results[cls_name]['lat_ratio'] = (
            (c3_mu - c4_mu) / (c3_mu + c4_mu)
            if (c3_mu + c4_mu) > 0 else 0
        )

    return results


#### load_raw_no_filter

In [12]:
def load_raw_no_filter(subject_id, runs=['R04', 'R08', 'R12']):
    """Load and concatenate raw EDF files, no filtering applied."""
    raw_list = []
    for run in runs:
        edf_path = raw_path / subject_id / f"{subject_id}{run}.edf"
        raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
        raw_list.append(raw)
    raw_concat = mne.concatenate_raws(raw_list, verbose=False)
    return raw_concat

#### extract_raw_epochs

In [13]:
def extract_raw_epochs(subject_id, tmin=0.3, tmax=1.0):
    """
    Load raw EDF, no filtering, extract motor imagery epochs.
    Returns X (n_trials, n_channels, n_times), y (n_trials,)
    for C3 and C4 channels only.
    """
    raw = load_raw_no_filter(subject_id)

    # Pick C3 and C4 using dot notation
    raw.pick(['C3..', 'C4..', 'Cp3.', 'Cp4.'])

    # Get events
    events, _ = mne.events_from_annotations(raw, verbose=False)

    # Extract epochs, no baseline correction, no filtering
    epochs = mne.Epochs(raw, events,
                        event_id=dict(T1=2, T2=3),
                        tmin=tmin, tmax=tmax,
                        baseline=None, preload=True,
                        verbose=False)

    X = epochs.get_data()  # (n_trials, n_channels, n_times)
    y = epochs.events[:, 2] - 2  # convert to 0/1

    print(f"{subject_id}: {X.shape}, labels {np.unique(y, return_counts=True)}")
    return X, y

### End of Functions

### Signal check on Preprocessed Data 
cannot be reported

#### Test and verification

In [14]:

# # Test on S007
# print("Testing S007...")
# s007_power = compute_psd_preprocessed('S007')
# for ch in channels_to_compare:
#     print(f"  {ch}: mu={s007_power[ch]['mu']:.4e}, beta={s007_power[ch]['beta']:.4e}")

In [15]:
all_results = {}
all_subjects_flat = []
subject_groups = {}

for group, subjects in subjects_to_check.items():
    for subj in subjects:
        all_subjects_flat.append(subj)
        subject_groups[subj] = group
        print(f"Processing {subj} ({group})...")
        all_results[subj] = compute_psd_preprocessed(subj)

# Print comparison tables
print("\n" + "="*70)
print("Mu power (8-13Hz) — preprocessed 16ch, window 0.3-1.0s")
print("="*70)
print(f"{'Subject':<10}{'Group':<20}", end="")
for ch in channels_to_compare:
    print(f"{ch:>12}", end="")
print()
print("-"*70)
for subj in all_subjects_flat:
    print(f"{subj:<10}{subject_groups[subj]:<20}", end="")
    for ch in channels_to_compare:
        print(f"{all_results[subj][ch]['mu']:>12.2f}", end="")
    print()

print("\n" + "="*70)
print("Beta power (13-30Hz) — preprocessed 16ch, window 0.3-1.0s")
print("="*70)
print(f"{'Subject':<10}{'Group':<20}", end="")
for ch in channels_to_compare:
    print(f"{ch:>12}", end="")
print()
print("-"*70)
for subj in all_subjects_flat:
    print(f"{subj:<10}{subject_groups[subj]:<20}", end="")
    for ch in channels_to_compare:
        print(f"{all_results[subj][ch]['beta']:>12.2f}", end="")
    print()

Processing S007 (top)...
Processing S085 (top)...
Processing S003 (negative_transfer)...
Processing S005 (negative_transfer)...
Processing S018 (negative_transfer)...
Processing S006 (group_a)...
Processing S037 (group_a)...
Processing S020 (group_b)...
Processing S066 (group_b)...
Processing S030 (neither)...
Processing S053 (neither)...
Processing S065 (neither)...

Mu power (8-13Hz) — preprocessed 16ch, window 0.3-1.0s
Subject   Group                         C3          C4         CP3         CP4
----------------------------------------------------------------------
S007      top                       694.05      782.05      677.78      662.13
S085      top                      1374.52     1204.20     1775.89     1705.63
S003      negative_transfer         673.78      772.71      825.10     1127.14
S005      negative_transfer         429.26      477.65      503.23      547.00
S018      negative_transfer         264.79      267.03      272.87      310.04
S006      group_a            

#### checks for signal 
Direction 2: does the signal differ between left and right class per channel

Direction 3: how consistent is the signal across trials, low CV means consistent

Direction 1: is the signal spatially lateralised in the expected direction

In [16]:
# Test on S007
print("Testing S007...")
s007_profile = compute_subject_signal_profile('S007')
for cls_name in ['left', 'right']:
    print(f"\nS007 {cls_name} class:")
    for ch in channels_to_compare:
        r = s007_profile[cls_name][ch]
        print(f"  {ch}: mu={r['mu_power']:.2f}, beta={r['beta_power']:.2f}, "
              f"mu_cv={r['mu_cv']:.3f}")
    print(f"  Lateralisation ratio: "
          f"{s007_profile[cls_name]['lateralisation_ratio']:.4f}")

Testing S007...

S007 left class:
  C3: mu=902.90, beta=131.85, mu_cv=0.899
  C4: mu=549.39, beta=117.81, mu_cv=1.051
  CP3: mu=937.06, beta=162.30, mu_cv=1.064
  CP4: mu=491.11, beta=87.09, mu_cv=0.868
  Lateralisation ratio: 0.2434

S007 right class:
  C3: mu=475.71, beta=130.56, mu_cv=0.996
  C4: mu=1025.29, beta=166.21, mu_cv=0.997
  CP3: mu=406.71, beta=112.01, mu_cv=0.820
  CP4: mu=840.93, beta=130.57, mu_cv=1.100
  Lateralisation ratio: -0.3661


##### All Subjects

In [17]:
all_profiles = {}
for subj in all_subjects_flat:
    print(f"Processing {subj} ({subject_groups[subj]})...")
    all_profiles[subj] = compute_subject_signal_profile(subj)

# Print lateralisation ratio comparison
print("\n" + "="*70)
print("Lateralisation ratio (C3-C4)/(C3+C4) per class")
print("Positive = C3 dominant, Negative = C4 dominant")
print("Expected: left class positive, right class negative")
print("="*70)
print(f"{'Subject':<10}{'Group':<22}{'Left class':>12}{'Right class':>12}")
print("-"*70)
for subj in all_subjects_flat:
    left_ratio = all_profiles[subj]['left']['lateralisation_ratio']
    right_ratio = all_profiles[subj]['right']['lateralisation_ratio']
    print(f"{subj:<10}{subject_groups[subj]:<22}{left_ratio:>12.4f}{right_ratio:>12.4f}")

# Print mu power comparison by class
print("\n" + "="*70)
print("C3 mu power - left class vs right class")
print("="*70)
print(f"{'Subject':<10}{'Group':<22}{'Left':>10}{'Right':>10}{'Diff':>10}")
print("-"*70)
for subj in all_subjects_flat:
    left_c3 = all_profiles[subj]['left']['C3']['mu_power']
    right_c3 = all_profiles[subj]['right']['C3']['mu_power']
    print(f"{subj:<10}{subject_groups[subj]:<22}"
          f"{left_c3:>10.2f}{right_c3:>10.2f}{left_c3-right_c3:>10.2f}")

# Print trial variability
print("\n" + "="*70)
print("Trial variability (coefficient of variation) - C3 mu, left class")
print("Higher CV = more variable signal across trials")
print("="*70)
print(f"{'Subject':<10}{'Group':<22}{'C3 mu CV':>10}{'C4 mu CV':>10}")
print("-"*70)
for subj in all_subjects_flat:
    c3_cv = all_profiles[subj]['left']['C3']['mu_cv']
    c4_cv = all_profiles[subj]['left']['C4']['mu_cv']
    print(f"{subj:<10}{subject_groups[subj]:<22}{c3_cv:>10.3f}{c4_cv:>10.3f}")

Processing S007 (top)...
Processing S085 (top)...
Processing S003 (negative_transfer)...
Processing S005 (negative_transfer)...
Processing S018 (negative_transfer)...
Processing S006 (group_a)...
Processing S037 (group_a)...
Processing S020 (group_b)...
Processing S066 (group_b)...
Processing S030 (neither)...
Processing S053 (neither)...
Processing S065 (neither)...

Lateralisation ratio (C3-C4)/(C3+C4) per class
Positive = C3 dominant, Negative = C4 dominant
Expected: left class positive, right class negative
Subject   Group                   Left class Right class
----------------------------------------------------------------------
S007      top                         0.2434     -0.3661
S085      top                         0.2923     -0.1725
S003      negative_transfer          -0.0885     -0.0451
S005      negative_transfer          -0.0188     -0.0869
S018      negative_transfer          -0.0649      0.0545
S006      group_a                    -0.0788     -0.2565
S037      gro

##### Summary


In [ ]:
print("="*80)
print("SIGNAL CHECK SUMMARY  12 subjects across 5 groups")
print("="*80)

print("\n1. LATERALISATION RATIO (C3-C4)/(C3+C4)")
print("   Positive = C3 dominant | Negative = C4 dominant")
print("   Expected: Left class positive, Right class negative")
print(f"\n{'Subject':<10}{'Group':<22}{'Left':>10}{'Right':>10}")
print("-"*55)
for subj in all_subjects_flat:
    left_r = all_profiles[subj]['left']['lateralisation_ratio']
    right_r = all_profiles[subj]['right']['lateralisation_ratio']
    print(f"{subj:<10}{subject_groups[subj]:<22}{left_r:>10.4f}{right_r:>10.4f}")

print("\n2. MU POWER BY CHANNEL AND CLASS")
print(f"\n{'Subject':<10}{'Group':<22}"
      f"{'C3 Left':>9}{'C3 Right':>9}{'C3 Diff':>9}"
      f"{'C4 Left':>9}{'C4 Right':>9}{'C4 Diff':>9}")
print("-"*85)
for subj in all_subjects_flat:
    c3_left = all_profiles[subj]['left']['C3']['mu_power']
    c3_right = all_profiles[subj]['right']['C3']['mu_power']
    c4_left = all_profiles[subj]['left']['C4']['mu_power']
    c4_right = all_profiles[subj]['right']['C4']['mu_power']
    c3_diff = c3_left - c3_right
    c4_diff = c4_right - c4_left
    print(f"{subj:<10}{subject_groups[subj]:<22}"
          f"{c3_left:>9.1f}{c3_right:>9.1f}{c3_diff:>9.1f}"
          f"{c4_left:>9.1f}{c4_right:>9.1f}{c4_diff:>9.1f}")

print("\n3. TRIAL VARIABILITY COEFFICIENT OF VARIATION")
print("   C3 CV left class | C4 CV right class")
print("   Higher value = more variable signal across trials")
print(f"\n{'Subject':<10}{'Group':<22}{'C3 CV (L)':>12}{'C4 CV (R)':>12}")
print("-"*58)
for subj in all_subjects_flat:
    left_cv = all_profiles[subj]['left']['C3']['mu_cv']
    right_cv = all_profiles[subj]['right']['C4']['mu_cv']
    print(f"{subj:<10}{subject_groups[subj]:<22}"
          f"{left_cv:>12.3f}{right_cv:>12.3f}")

print("\n4. ABSOLUTE MU POWER KEY CHANNELS")
print("   C3 left class | C4 right class | CP3 left class | CP4 right class")
print(f"\n{'Subject':<10}{'Group':<22}{'C3(L)':>8}{'C4(R)':>8}"
      f"{'CP3(L)':>8}{'CP4(R)':>8}")
print("-"*68)
for subj in all_subjects_flat:
    c3_left = all_profiles[subj]['left']['C3']['mu_power']
    c4_right = all_profiles[subj]['right']['C4']['mu_power']
    cp3_left = all_profiles[subj]['left']['CP3']['mu_power']
    cp4_right = all_profiles[subj]['right']['CP4']['mu_power']
    print(f"{subj:<10}{subject_groups[subj]:<22}"
          f"{c3_left:>8.1f}{c4_right:>8.1f}"
          f"{cp3_left:>8.1f}{cp4_right:>8.1f}")

SIGNAL CHECK SUMMARY — 12 subjects across 5 groups

1. LATERALISATION RATIO (C3-C4)/(C3+C4)
   Positive = C3 dominant | Negative = C4 dominant
   Expected: Left class positive, Right class negative

Subject   Group                       Left     Right
-------------------------------------------------------
S007      top                       0.2434   -0.3661
S085      top                       0.2923   -0.1725
S003      negative_transfer        -0.0885   -0.0451
S005      negative_transfer        -0.0188   -0.0869
S018      negative_transfer        -0.0649    0.0545
S006      group_a                  -0.0788   -0.2565
S037      group_a                   0.0364   -0.1107
S020      group_b                   0.0144    0.1106
S066      group_b                   0.0243   -0.1424
S030      neither                  -0.0693    0.0853
S053      neither                   0.1476    0.1594
S065      neither                  -0.1368   -0.0042

2. MU POWER BY CHANNEL AND CLASS

Subject   Group      

## Raw signal

In [ ]:
# raw_path = Path("../Data/raw/physionet_mi_data/eeg-motor-movementimagery-dataset")

# Same 12 subjects as preprocessed check
subjects_to_check = {
    'top': ['S007', 'S085'],
    'negative_transfer': ['S003', 'S005', 'S018'],
    'group_a': ['S006', 'S037'],
    'group_b': ['S020', 'S066'],
    'neither': ['S030', 'S053', 'S065']
}

# Test load S007 raw, no filtering
raw = mne.io.read_raw_edf(
    raw_path / 'S007' / 'S007R04.edf',
    preload=True, verbose=False)

print("Channels:", len(raw.ch_names))
print("Sampling rate:", raw.info['sfreq'])
print("Duration:", raw.times[-1])

Channels: 64
Sampling rate: 160.0
Duration: 124.99375


In [ ]:
# events from annotations
events, event_dict = mne.events_from_annotations(raw, verbose=False)
print("Event dict:", event_dict)
print("Unique event ids:", np.unique(events[:, 2]))
print("Total events:", len(events))

Event dict: {np.str_('T0'): 1, np.str_('T1'): 2, np.str_('T2'): 3}
Unique event ids: [1 2 3]
Total events: 30


In [21]:
# Test on S007
X_s007, y_s007 = extract_raw_epochs('S007')
print(f"Data range: min={X_s007.min():.2e}, max={X_s007.max():.2e}")

S007: (45, 4, 113), labels (array([0, 1]), array([23, 22]))
Data range: min=-1.19e-04, max=1.44e-04


In [22]:
# Test on S007
print("Testing S007 full 16 channel raw signal...")
s007_full = compute_raw_psd('S007')
print(f"Channels computed: {list(s007_full['left'].keys())}")
print(f"Left lat ratio: {s007_full['left']['lat_ratio']:.4f}")
print(f"Right lat ratio: {s007_full['right']['lat_ratio']:.4f}")
print(f"Left C3 mu: {s007_full['left']['C3']['mu']:.4e}")
print(f"Left CP3 mu: {s007_full['left']['CP3']['mu']:.4e}")

Testing S007 full 16 channel raw signal...
Channels computed: ['C3', 'Cz', 'C4', 'FC3', 'FCz', 'CP3', 'CPz', 'CP4', 'CP5', 'CP6', 'P3', 'P4', 'C5', 'C6', 'P5', 'P6', 'lat_ratio']
Left lat ratio: 0.2575
Right lat ratio: 0.0138
Left C3 mu: 2.5396e-07
Left CP3 mu: 2.7427e-07


In [23]:
all_raw_profiles = {}
for group, subjects in subjects_to_check.items():
    for subj in subjects:
        print(f"Processing {subj} ({group})...")
        all_raw_profiles[subj] = compute_raw_psd(subj)

# Print lateralisation summary
print("\n" + "="*70)
print("RAW SIGNAL - Lateralisation ratio (no filtering)")
print(f"{'Subject':<10}{'Group':<22}{'Left':>10}{'Right':>10}")
print("-"*55)
for group in ['top', 'negative_transfer', 'group_a', 'group_b', 'neither']:
    for subj in subjects_to_check[group]:
        left_r = all_raw_profiles[subj]['left']['lat_ratio']
        right_r = all_raw_profiles[subj]['right']['lat_ratio']
        print(f"{subj:<10}{group:<22}{left_r:>10.4f}{right_r:>10.4f}")

Processing S007 (top)...
Processing S085 (top)...
Processing S003 (negative_transfer)...
Processing S005 (negative_transfer)...
Processing S018 (negative_transfer)...
Processing S006 (group_a)...
Processing S037 (group_a)...
Processing S020 (group_b)...
Processing S066 (group_b)...
Processing S030 (neither)...
Processing S053 (neither)...
Processing S065 (neither)...

RAW SIGNAL - Lateralisation ratio (no filtering)
Subject   Group                       Left     Right
-------------------------------------------------------
S007      top                       0.2575    0.0138
S085      top                       0.1434   -0.2360
S003      negative_transfer         0.0204    0.0875
S005      negative_transfer         0.1956    0.0543
S018      negative_transfer         0.0078    0.0736
S006      group_a                   0.0820    0.0831
S037      group_a                   0.1608    0.0780
S020      group_b                   0.1797    0.1034
S066      group_b                   0.2261    0

In [24]:
print("\n" + "="*70)
print("RAW SIGNAL - C3 mu power difference (left - right class)")
print(f"{'Subject':<10}{'Group':<22}{'C3 Left':>12}{'C3 Right':>12}{'Diff':>12}")
print("-"*70)
for group in ['top', 'negative_transfer', 'group_a', 'group_b', 'neither']:
    for subj in subjects_to_check[group]:
        c3_left = all_raw_profiles[subj]['left']['C3']['mu']
        c3_right = all_raw_profiles[subj]['right']['C3']['mu']
        diff = c3_left - c3_right
        print(f"{subj:<10}{group:<22}"
              f"{c3_left:>12.8f}{c3_right:>12.8f}{diff:>12.8f}")


RAW SIGNAL - C3 mu power difference (left - right class)
Subject   Group                      C3 Left    C3 Right        Diff
----------------------------------------------------------------------
S007      top                     0.00000025  0.00000018  0.00000007
S085      top                     0.00000022  0.00000018  0.00000004
S003      negative_transfer       0.00000050  0.00000038  0.00000013
S005      negative_transfer       0.00000009  0.00000006  0.00000003
S018      negative_transfer       0.00000001  0.00000001 -0.00000000
S006      group_a                 0.00000004  0.00000004  0.00000001
S037      group_a                 0.00000001  0.00000001  0.00000000
S020      group_b                 0.00000001  0.00000002 -0.00000001
S066      group_b                 0.00000021  0.00000017  0.00000004
S030      neither                 0.00000003  0.00000004 -0.00000000
S053      neither                 0.00000052  0.00000034  0.00000018
S065      neither                 0.0000002

In [25]:
print("\n" + "="*70)
print("RAW SIGNAL SUMMARY - All metrics")
print("="*70)
print(f"{'Subject':<10}{'Group':<22}{'Lat(L)':>8}{'Lat(R)':>8}"
      f"{'C3 Diff':>12}{'C4 Diff':>12}")
print("-"*72)
for group in ['top', 'negative_transfer', 'group_a', 'group_b', 'neither']:
    for subj in subjects_to_check[group]:
        left_r = all_raw_profiles[subj]['left']['lat_ratio']
        right_r = all_raw_profiles[subj]['right']['lat_ratio']
        c3_left = all_raw_profiles[subj]['left']['C3']['mu']
        c3_right = all_raw_profiles[subj]['right']['C3']['mu']
        c4_left = all_raw_profiles[subj]['left']['C4']['mu']
        c4_right = all_raw_profiles[subj]['right']['C4']['mu']
        c3_diff = c3_left - c3_right
        c4_diff = c4_right - c4_left
        print(f"{subj:<10}{group:<22}"
              f"{left_r:>8.4f}{right_r:>8.4f}"
              f"{c3_diff:>12.8f}{c4_diff:>12.8f}")


RAW SIGNAL SUMMARY - All metrics
Subject   Group                   Lat(L)  Lat(R)     C3 Diff     C4 Diff
------------------------------------------------------------------------
S007      top                     0.2575  0.0138  0.00000007  0.00000003
S085      top                     0.1434 -0.2360  0.00000004  0.00000013
S003      negative_transfer       0.0204  0.0875  0.00000013 -0.00000017
S005      negative_transfer       0.1956  0.0543  0.00000003 -0.00000001
S018      negative_transfer       0.0078  0.0736 -0.00000000  0.00000000
S006      group_a                 0.0820  0.0831  0.00000001 -0.00000000
S037      group_a                 0.1608  0.0780  0.00000000  0.00000000
S020      group_b                 0.1797  0.1034 -0.00000001  0.00000001
S066      group_b                 0.2261  0.0784  0.00000004  0.00000001
S030      neither                 0.1244  0.1510 -0.00000000  0.00000000
S053      neither                 0.3758  0.2193  0.00000018 -0.00000002
S065      neither

### Raw signal correlation with kappa

In [26]:
with open(results_path / 'loso_results_cdan_16ch_RETRAIN.json') as f:
    binary_results = json.load(f)

binary_kappa = {
    subj: binary_results[subj]['kappa_stopping']['final_kappa']
    for subj in valid_subjects
}

print(f"Loaded: {len(binary_kappa)} subjects")
print(f"Kappa range: {min(binary_kappa.values()):.4f} to {max(binary_kappa.values()):.4f}")

Loaded: 106 subjects
Kappa range: -0.0177 to 1.0000


In [ ]:
print("Computing raw signal for all 106 subjects...")
start = time.time()

all_raw_106 = {}
for subj in valid_subjects:
    all_raw_106[subj] = compute_raw_psd(subj)

# elapsed = time.time() - start
# print(f"Done. Time: {elapsed:.1f} seconds")

# Correlate all 16 channels with kappa
kappa_list = [binary_kappa[s] for s in valid_subjects]
print(f"\nRaw signal correlation with kappa (106 subjects)")
print(f"{'Channel':<8}{'Mu Diff r':>12}{'Lat Ratio r':>14}")
print("="*36)

for ch in ch16_list:
    # mu power difference left minus right
    ch_diff = [all_raw_106[s]['left'][ch]['mu'] -
               all_raw_106[s]['right'][ch]['mu']
               for s in valid_subjects]
    r_diff = np.corrcoef(ch_diff, kappa_list)[0, 1]

    # channel specific lateralisation: ch_left vs ch_right
    ch_lat = [(all_raw_106[s]['left'][ch]['mu'] -
               all_raw_106[s]['right'][ch]['mu']) /
              (all_raw_106[s]['left'][ch]['mu'] +
               all_raw_106[s]['right'][ch]['mu'])
              if (all_raw_106[s]['left'][ch]['mu'] +
                  all_raw_106[s]['right'][ch]['mu']) > 0
              else 0
              for s in valid_subjects]
    r_lat = np.corrcoef(ch_lat, kappa_list)[0, 1]

    print(f"{ch:<8}{r_diff:>12.4f}{r_lat:>14.4f}")

# Overall C3 vs C4 lateralisation
print(f"\nC3 vs C4 lateralisation ratio correlation with kappa:")
lat_left = [all_raw_106[s]['left']['lat_ratio'] for s in valid_subjects]
lat_right = [all_raw_106[s]['right']['lat_ratio'] for s in valid_subjects]
r_lat_left = np.corrcoef(lat_left, kappa_list)[0, 1]
r_lat_right = np.corrcoef(lat_right, kappa_list)[0, 1]
print(f"  Left class (C3 vs C4): r={r_lat_left:.4f}")
print(f"  Right class (C3 vs C4): r={r_lat_right:.4f}")

Computing raw signal for all 106 subjects...
Done. Time: 3.1 seconds

Raw signal correlation with kappa (106 subjects)
Channel    Mu Diff r   Lat Ratio r
C3            0.1226        0.1999
Cz           -0.0685       -0.0422
C4            0.0194       -0.2490
FC3          -0.0452        0.0067
FCz          -0.0947       -0.0556
CP3           0.2163        0.2996
CPz           0.0422        0.0802
CP4          -0.1903       -0.2265
CP5           0.1171        0.2451
CP6          -0.2186       -0.2805
P3            0.2092        0.3242
P4           -0.1858       -0.2128
C5            0.0527        0.1542
C6           -0.1744       -0.2491
P5            0.1938        0.3352
P6           -0.2085       -0.2368

C3 vs C4 lateralisation ratio correlation with kappa:
  Left class (C3 vs C4): r=0.1556
  Right class (C3 vs C4): r=-0.2261


### raw signal lateralisation between top and negative transfer subjects

In [28]:
results_path = Path('../results')

# load kappas
with open(results_path / 'loso_results_cdan_16ch_RETRAIN.json') as f:
    retrain_data = json.load(f)

binary_kappas = {s: retrain_data[s]['kappa_stopping']['final_kappa']
                 for s in retrain_data}

# get score groups
score_groups, group_boundaries = assign_score_groups(binary_kappas)

top_subjects = [s for s, g in score_groups.items() if g == 'high']
neg_subjects = [s for s, g in score_groups.items() if g == 'negative_transfer']

print(f"Top group: {len(top_subjects)} subjects")
print(f"Negative transfer group: {len(neg_subjects)} subjects")

# use raw signal lateralisation from all_raw_106
# if not in memory, rerun cell 24 first
top_raw_lat = np.array([
    (abs(all_raw_106[s]['left']['lat_ratio']) +
     abs(all_raw_106[s]['right']['lat_ratio'])) / 2
    for s in top_subjects if s in all_raw_106
])

neg_raw_lat = np.array([
    (abs(all_raw_106[s]['left']['lat_ratio']) +
     abs(all_raw_106[s]['right']['lat_ratio'])) / 2
    for s in neg_subjects if s in all_raw_106
])

print(f"Top group mean raw lateralisation: {top_raw_lat.mean():.4f}")
print(f"Negative transfer mean raw lateralisation: {neg_raw_lat.mean():.4f}")

stat, p = stats.mannwhitneyu(top_raw_lat, neg_raw_lat, alternative='greater')
print(f"Mann-Whitney U: stat={stat:.1f}, p={p:.6f}")

Top group: 30 subjects
Negative transfer group: 3 subjects
Top group mean raw lateralisation: 0.1487
Negative transfer mean raw lateralisation: 0.0732
Mann-Whitney U: stat=76.0, p=0.026943


In [29]:
# Hypothesis Test: Raw signal lateralisation: Top vs Negative Transfer
# Mann-Whitney U test on raw unfiltered EEG lateralisation ratios

top_raw_lat = np.array([
    (abs(all_raw_106[s]['left']['lat_ratio']) +
     abs(all_raw_106[s]['right']['lat_ratio'])) / 2
    for s in top_subjects if s in all_raw_106
])

neg_raw_lat = np.array([
    (abs(all_raw_106[s]['left']['lat_ratio']) +
     abs(all_raw_106[s]['right']['lat_ratio'])) / 2
    for s in neg_subjects if s in all_raw_106
])

print(f"Top group n={len(top_raw_lat)}, mean raw lateralisation: {top_raw_lat.mean():.4f}")
print(f"Negative transfer n={len(neg_raw_lat)}, mean raw lateralisation: {neg_raw_lat.mean():.4f}")

stat, p = stats.mannwhitneyu(top_raw_lat, neg_raw_lat, alternative='greater')
print(f"Mann-Whitney U: stat={stat:.1f}, p={p:.6f}")

Top group n=30, mean raw lateralisation: 0.1487
Negative transfer n=3, mean raw lateralisation: 0.0732
Mann-Whitney U: stat=76.0, p=0.026943
